In [106]:
import pandas as pd

In [107]:
gdsc_bulk = pd.read_excel("../data/GDSC2_fitted_dose_response_27Oct23.xlsx")

# change CELL_LINE_NAME column to remove "-" from values
gdsc_bulk["CELL_LINE_NAME"] = gdsc_bulk["CELL_LINE_NAME"].str.replace("-", "")
gdsc_bulk.head()

,DATASET,NLME_RESULT_ID,NLME_CURVE_ID,CELL_LINE_NAME,SANGER_MODEL_ID,CANCER_TYPE,DRUG_ID,DRUG_NAME,PUTATIVE_TARGET,PATHWAY_NAME,MIN_CONC,MAX_CONC,LN_IC50,AUC,RMSE,Z_SCORE
0,GDSC2,343,15946310,PFSK1,SIDM01132,Other Solid Cancers,1003,Camptothecin,TOP1,DNA replication,0.0001,0.1,-1.463887,0.930220,0.089052,0.433123
1,GDSC2,343,15946548,A673,SIDM00848,Ewing's Sarcoma,1003,Camptothecin,TOP1,DNA replication,0.0001,0.1,-4.869455,0.614970,0.111351,-1.421100
2,GDSC2,343,15946830,ES5,SIDM00263,Ewing's Sarcoma,1003,Camptothecin,TOP1,DNA replication,0.0001,0.1,-3.360586,0.791072,0.142855,-0.599569
3,GDSC2,343,15947087,ES7,SIDM00269,Ewing's Sarcoma,1003,Camptothecin,TOP1,DNA replication,0.0001,0.1,-5.044940,0.592660,0.135539,-1.516647
4,GDSC2,343,15947369,EW11,SIDM00203,Ewing's Sarcoma,1003,Camptothecin,TOP1,DNA replication,0.0001,0.1,-3.741991,0.734047,0.128059,-0.807232


In [108]:
scp542_scrna = pd.read_csv("/Users/selin/Desktop/OncoTox/scRNAseq_SCP542/metadata/Metadata.txt", sep="\t", skiprows=[1])
scp542_scrna["Cell_line"] = scp542_scrna["Cell_line"].str.split("_").str[0]
scp542_scrna.head()

/var/folders/6k/gr_1_h_97154rq71pm_q3jn40000gn/T/ipykernel_53796/2389764182.py:1: DtypeWarning: Columns (0: Pool_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  scp542_scrna = pd.read_csv("/Users/selin/Desktop/OncoTox/scRNAseq_SCP542/metadata/Metadata.txt", sep="\t", skiprows=[1])


,NAME,Cell_line,Pool_ID,Cancer_type,Genes_expressed,Discrete_cluster_minpts5_eps1.8,Discrete_cluster_minpts5_eps1.5,Discrete_cluster_minpts5_eps1.2,CNA_subclone,SkinPig_score,...,EMTII_score,EMTIII_score,IFNResp_score,p53Sen_score,EpiSen_score,StressResp_score,ProtMatu_score,ProtDegra_score,G1/S_score,G2/M_score
0,AAACCTGAGACATAAC-1-18,NCIH2126,18,Lung Cancer,4318,NaN,NaN,NaN,NaN,0.166,...,-0.935,-0.935,0.130,0.619,1.869,-0.004,0.805,0.896,0.424,-1.125
1,AACGTTGTCACCCGAG-1-18,NCIH2126,18,Lung Cancer,5200,NaN,NaN,NaN,NaN,-0.213,...,-1.027,-1.027,0.066,1.049,1.267,0.252,1.299,1.610,0.624,-0.048
2,AACTGGTAGACACGAC-1-18,NCIH2126,18,Lung Cancer,4004,NaN,NaN,NaN,NaN,-0.101,...,-0.677,-0.677,0.304,0.822,2.401,0.141,0.451,1.225,-0.795,0.064
3,AACTGGTAGGGCTTGA-1-18,NCIH2126,18,Lung Cancer,4295,NaN,NaN,NaN,NaN,-0.014,...,-0.735,-0.735,0.094,0.834,2.282,0.150,0.267,0.892,-0.238,1.118
4,AACTGGTAGTACTTGC-1-18,NCIH2126,18,Lung Cancer,4842,NaN,NaN,NaN,NaN,0.006,...,-0.821,-0.821,0.034,0.960,1.400,-0.012,-0.276,-0.428,0.267,0.791


In [109]:
# Reusable helper to compare SCP542 coverage in another cell-line dataset

def build_unique_with_norm(df, source_col, norm_col):
    return (
        df.assign(**{norm_col: df[source_col].astype(str).str.strip().str.lower()})
        .drop_duplicates(subset=norm_col)
        .copy()
    )


def report_scp542_missing(reference_unique, reference_norm_col, reference_label):
    print(f"Unique {reference_label} cell lines: {reference_unique[reference_norm_col].nunique()}")
    print(f"Unique scp542 cell lines: {scp542_unique['CCL_NAME_NORM'].nunique()}")

    ref_norm_set = set(reference_unique[reference_norm_col])
    missing_scp542 = scp542_unique.loc[
        ~scp542_unique["CCL_NAME_NORM"].isin(ref_norm_set)
    ].copy()

    print(f"\nUnique scp542 cell lines missing from {reference_label}: {len(missing_scp542)}")
    display(missing_scp542[["Cell_line"]].sort_values("Cell_line").head(50))
    return missing_scp542


scp542_unique = build_unique_with_norm(scp542_scrna, "Cell_line", "CCL_NAME_NORM")
gdsc_unique = build_unique_with_norm(gdsc_bulk, "CELL_LINE_NAME", "CELL_LINE_NAME_NORM")
missing_scp542 = report_scp542_missing(gdsc_unique, "CELL_LINE_NAME_NORM", "bulk GDSC")


Unique bulk GDSC cell lines: 967
Unique scp542 cell lines: 198

Unique scp542 cell lines missing from bulk GDSC: 65


,Cell_line
52358,93VU
17101,ACCMESO1
44729,BICR16
21592,BICR56
6120,BICR6
48519,CAKI2
7100,CCFSTTG1
36215,CL34
2293,COLO741
10016,COV434


In [110]:
ctrp = pd.read_csv("/Users/selin/Desktop/OncoTox/metadata/CTRPv2.0_2015_ctd2_ExpandedDataset/v20.meta.per_cell_line.txt", sep="\t")
ctrp.head()

,master_ccl_id,ccl_name,ccl_availability,ccle_primary_site,ccle_primary_hist,ccle_hist_subtype_1
0,1,697,ccle;public,haematopoietic_and_lymphoid_tissue,lymphoid_neoplasm,acute_lymphoblastic_B_cell_leukaemia
1,3,5637,ccle;public,urinary_tract,carcinoma,NaN
2,4,2313287,ccle;public,stomach,carcinoma,adenocarcinoma
3,5,1321N1,ccle,central_nervous_system,glioma,astrocytoma
4,6,143B,ccle,bone,osteosarcoma,NaN


In [111]:
# Compare unique cell lines (case-insensitive) between CTRP and SCP542
ctrp_unique = build_unique_with_norm(ctrp, "ccl_name", "CELL_LINE_NAME_NORM")
missing_scp542 = report_scp542_missing(ctrp_unique, "CELL_LINE_NAME_NORM", "CTRP")


Unique CTRP cell lines: 1107
Unique scp542 cell lines: 198

Unique scp542 cell lines missing from CTRP: 8


,Cell_line
52358,93VU
53188,JHU006
51641,JHU011
49471,JHU029
31688,NCIH2077
40844,NCIH292
48879,SCC47
52930,SCC90


In [112]:
# /Users/selin/Desktop/OncoTox/metadata/PRISM_REPURPOSED/Repurposing_Public_24Q2_Cell_Line_Meta_Data.csv
prism = pd.read_csv("/Users/selin/Desktop/OncoTox/metadata/PRISM_REPURPOSED/Repurposing_Public_24Q2_Cell_Line_Meta_Data.csv")
prism["ccle_name"] = prism["ccle_name"].str.split("_").str[0]
prism.head()

,ccle_name,row_id,pool_id,culture,depmap_id,screen
0,KYSE510,ACH-000824::P107::PR500A::REP1M,P107,PR500A,ACH-000824,REP1M
1,HEC1A,ACH-000954::P107::PR500A::REP1M,P107,PR500A,ACH-000954,REP1M
2,MIAPACA2,ACH-000601::P101::PR500A::REP1M,P101,PR500A,ACH-000601,REP1M
3,SW620,ACH-000651::P108::PR500A::REP1M,P108,PR500A,ACH-000651,REP1M
4,SKHEP1,ACH-000361::P108::PR500A::REP1M,P108,PR500A,ACH-000361,REP1M


In [113]:
# Compare unique cell lines (case-insensitive) between PRISM and SCP542
prism_unique = build_unique_with_norm(prism, "ccle_name", "CELL_LINE_NAME_NORM")
missing_scp542 = report_scp542_missing(prism_unique, "CELL_LINE_NAME_NORM", "prism")


Unique prism cell lines: 915
Unique scp542 cell lines: 198

Unique scp542 cell lines missing from prism: 16


,Cell_line
52358,93VU
48519,CAKI2
44527,HCC366
53188,JHU006
51641,JHU011
49471,JHU029
30535,KPL1
4548,KPNSI9S
46212,NCIH2073
45863,OAW28


In [114]:
# Summary table: SCP542 coverage across datasets

def normalized_cell_line_set(df, col, split_on_underscore=False):
    series = df[col].dropna().astype(str).str.strip().str.lower()
    if split_on_underscore:
        series = series.str.split("_").str[0]
    series = series[series != ""]
    return set(series.unique())


# Match preprocessing used earlier in the notebook for SCP542 names
scp542_set = normalized_cell_line_set(scp542_scrna, "Cell_line", split_on_underscore=True)
gdsc_set = normalized_cell_line_set(gdsc_bulk, "CELL_LINE_NAME")
ctrp_set = normalized_cell_line_set(ctrp, "ccl_name")
prism_set = normalized_cell_line_set(prism, "ccle_name", split_on_underscore=True)

# Drug counts from dedicated compound metadata files
ctrp_compounds = pd.read_csv(
    "/Users/selin/Desktop/OncoTox/metadata/CTRPv2.0_2015_ctd2_ExpandedDataset/v20.meta.per_compound.txt",
    sep="\t",
)
prism_compounds = pd.read_csv(
    "/Users/selin/Desktop/OncoTox/metadata/PRISM_REPURPOSED/Repurposing_Public_24Q2_Extended_Primary_Compound_List.csv"
)

gdsc_drug_count = int(gdsc_bulk["DRUG_ID"].nunique())
ctrp_drug_count = int(ctrp_compounds["master_cpd_id"].nunique())
prism_drug_count = int(prism_compounds["Drug.Name"].nunique())

summary_df = pd.DataFrame([
    {
        "Dataset": "GDSC",
        "Total Cell Lines": len(gdsc_set),
        "Overlap with SCP542": len(gdsc_set & scp542_set),
        "Missing Lines": len(scp542_set - gdsc_set),
        "Target Metric": "LN_IC50 / AUC",
        "Drug Count": gdsc_drug_count,
    },
    {
        "Dataset": "CTRPv2",
        "Total Cell Lines": len(ctrp_set),
        "Overlap with SCP542": len(ctrp_set & scp542_set),
        "Missing Lines": len(scp542_set - ctrp_set),
        "Target Metric": "Viability / AUC",
        "Drug Count": ctrp_drug_count,
    },
    {
        "Dataset": "PRISM",
        "Total Cell Lines": len(prism_set),
        "Overlap with SCP542": len(prism_set & scp542_set),
        "Missing Lines": len(scp542_set - prism_set),
        "Target Metric": "Viability",
        "Drug Count": prism_drug_count,
    },
])

display(summary_df)


,Dataset,Total Cell Lines,Overlap with SCP542,Missing Lines,Target Metric,Drug Count
0,GDSC,967,133,65,LN_IC50 / AUC,295
1,CTRPv2,1107,190,8,Viability / AUC,545
2,PRISM,915,182,16,Viability,6575


In [115]:
# Build a unified drug/compound catalog across GDSC, CTRPv2, and PRISM

# GDSC compounds from fitted dose-response table
gdsc_drugs = (
    gdsc_bulk[
        ["DRUG_ID", "DRUG_NAME", "PUTATIVE_TARGET", "PATHWAY_NAME", "DATASET"]
    ]
    .dropna(subset=["DRUG_ID"])
    .drop_duplicates(subset=["DRUG_ID"])
    .rename(
        columns={
            "DRUG_ID": "identifier",
            "DRUG_NAME": "compound_name",
            "PUTATIVE_TARGET": "target",
            "PATHWAY_NAME": "moa_or_pathway",
            "DATASET": "source_subdataset",
        }
    )
)
gdsc_drugs["dataset"] = "GDSC"
gdsc_drugs["identifier"] = gdsc_drugs["identifier"].astype(str)
gdsc_drugs["source_id_type"] = "DRUG_ID"

# CTRPv2 compounds from compound metadata
ctrp_compounds = pd.read_csv(
    "/Users/selin/Desktop/OncoTox/metadata/CTRPv2.0_2015_ctd2_ExpandedDataset/v20.meta.per_compound.txt",
    sep="\t",
)
ctrp_drugs = (
    ctrp_compounds[
        [
            "master_cpd_id",
            "cpd_name",
            "broad_cpd_id",
            "cpd_status",
            "gene_symbol_of_protein_target",
            "target_or_activity_of_compound",
            "source_name",
            "source_catalog_id",
            "cpd_smiles",
            "inclusion_rationale",
            "top_test_conc_umol",
        ]
    ]
    .dropna(subset=["master_cpd_id"])
    .drop_duplicates(subset=["master_cpd_id"])
    .rename(
        columns={
            "master_cpd_id": "identifier",
            "cpd_name": "compound_name",
            "broad_cpd_id": "alt_identifier",
            "cpd_status": "compound_status",
            "gene_symbol_of_protein_target": "target",
            "target_or_activity_of_compound": "moa_or_pathway",
            "source_name": "vendor_or_source",
            "source_catalog_id": "vendor_catalog_id",
            "cpd_smiles": "smiles",
            "inclusion_rationale": "notes",
            "top_test_conc_umol": "top_test_conc_umol",
        }
    )
)
ctrp_drugs["dataset"] = "CTRPv2"
ctrp_drugs["identifier"] = ctrp_drugs["identifier"].astype(str)
ctrp_drugs["source_id_type"] = "master_cpd_id"

# PRISM compounds from repurposing compound list
prism_compounds = pd.read_csv(
    "/Users/selin/Desktop/OncoTox/metadata/PRISM_REPURPOSED/Repurposing_Public_24Q2_Extended_Primary_Compound_List.csv"
)
prism_drugs = (
    prism_compounds[
        ["Drug.Name", "repurposing_target", "MOA", "IDs", "Synonyms", "screen", "dose"]
    ]
    .dropna(subset=["Drug.Name"])
    .drop_duplicates(subset=["Drug.Name"])
    .rename(
        columns={
            "Drug.Name": "identifier",
            "repurposing_target": "target",
            "MOA": "moa_or_pathway",
            "IDs": "alt_identifier",
            "Synonyms": "synonyms",
            "screen": "screen",
            "dose": "dose",
        }
    )
)
prism_drugs["compound_name"] = prism_drugs["identifier"]
prism_drugs["dataset"] = "PRISM"
prism_drugs["identifier"] = prism_drugs["identifier"].astype(str)
prism_drugs["source_id_type"] = "Drug.Name"

# Align columns and concatenate
all_drugs_df = pd.concat([gdsc_drugs, ctrp_drugs, prism_drugs], ignore_index=True, sort=False)

# Keep identifier first, dataset second
base_cols = ["identifier", "dataset"]
other_cols = [c for c in all_drugs_df.columns if c not in base_cols]
all_drugs_df = all_drugs_df[base_cols + other_cols]

# Add a normalized name helper for future duplicate review
if "compound_name" in all_drugs_df.columns:
    all_drugs_df["compound_name_norm"] = (
        all_drugs_df["compound_name"].astype(str).str.strip().str.lower()
    )

output_csv = "../data/drug/all_sources_drug_catalog.csv"
all_drugs_df.to_csv(output_csv, index=False)

print(f"Saved drug catalog to: {output_csv}")
print(f"Rows: {len(all_drugs_df)}")
print("Rows by dataset:")
print(all_drugs_df["dataset"].value_counts())

display(all_drugs_df.head())


Saved drug catalog to: ../data/drug/all_sources_drug_catalog.csv
Rows: 7415
Rows by dataset:
dataset
PRISM     6575
CTRPv2     545
GDSC       295
Name: count, dtype: int64


,identifier,dataset,compound_name,target,moa_or_pathway,source_subdataset,source_id_type,alt_identifier,compound_status,vendor_or_source,vendor_catalog_id,smiles,notes,top_test_conc_umol,synonyms,screen,dose,compound_name_norm
0,1003,GDSC,Camptothecin,TOP1,DNA replication,GDSC2,DRUG_ID,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,camptothecin
1,1004,GDSC,Vinblastine,Microtubule destabiliser,Mitosis,GDSC2,DRUG_ID,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,vinblastine
2,1005,GDSC,Cisplatin,DNA crosslinker,DNA replication,GDSC2,DRUG_ID,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cisplatin
3,1006,GDSC,Cytarabine,Antimetabolite,Other,GDSC2,DRUG_ID,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cytarabine
4,1007,GDSC,Docetaxel,Microtubule stabiliser,Mitosis,GDSC2,DRUG_ID,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,docetaxel


In [116]:
# Analyze overlap strategies across datasets using the unified drug catalog
import re


def extract_canonical_brd(value):
    if pd.isna(value):
        return pd.NA
    match = re.search(r"BRD-[A-Z]\d{8}", str(value))
    return match.group(0) if match else pd.NA


# Work from the unified table created in the previous cell
drug_df = all_drugs_df.copy()
drug_df["compound_name_norm"] = drug_df["compound_name"].astype(str).str.strip().str.lower()
drug_df["brd_canonical"] = drug_df["alt_identifier"].map(extract_canonical_brd)

# Cache dataset-level tables/sets for reuse in this and later cells
dataset_frames = {
    ds: sub[
        ["identifier", "dataset", "compound_name", "compound_name_norm", "target", "moa_or_pathway", "brd_canonical"]
    ].drop_duplicates()
    for ds, sub in drug_df.groupby("dataset")
}
dataset_sets = {
    ds: set(frame["compound_name_norm"].dropna().unique()) for ds, frame in dataset_frames.items()
}
datasets = sorted(dataset_frames.keys())

# Pairwise overlap summary (name-based and BRD-based)
summary_rows = []
for i, ds1 in enumerate(datasets):
    for ds2 in datasets[i + 1:]:
        a = dataset_frames[ds1]
        b = dataset_frames[ds2]
        summary_rows.append(
            {
                "dataset_a": ds1,
                "dataset_b": ds2,
                "name_overlap_count": len(dataset_sets[ds1] & dataset_sets[ds2]),
                "brd_overlap_count": len(set(a["brd_canonical"].dropna()) & set(b["brd_canonical"].dropna())),
            }
        )

overlap_summary_df = pd.DataFrame(summary_rows)
print("Pairwise overlap summary:")
display(overlap_summary_df)

# Build candidate overlap pairs by exact normalized name
name_candidate_frames = []
for i, ds1 in enumerate(datasets):
    for ds2 in datasets[i + 1:]:
        a = dataset_frames[ds1]
        b = dataset_frames[ds2]

        merged = a.merge(b, on="compound_name_norm", suffixes=("_a", "_b"), how="inner")
        if merged.empty:
            continue

        out = pd.DataFrame(
            {
                "dataset_a": merged["dataset_a"],
                "identifier_a": merged["identifier_a"],
                "compound_name_a": merged["compound_name_a"],
                "dataset_b": merged["dataset_b"],
                "identifier_b": merged["identifier_b"],
                "compound_name_b": merged["compound_name_b"],
                "compound_name_norm": merged["compound_name_norm"],
                "brd_canonical": merged["brd_canonical_a"].fillna(merged["brd_canonical_b"]),
                "match_method": "name_exact",
                "target_a": merged["target_a"],
                "target_b": merged["target_b"],
                "moa_or_pathway_a": merged["moa_or_pathway_a"],
                "moa_or_pathway_b": merged["moa_or_pathway_b"],
            }
        )
        name_candidate_frames.append(out)

name_candidates_df = (
    pd.concat(name_candidate_frames, ignore_index=True)
    if name_candidate_frames
    else pd.DataFrame()
)

# Build high-confidence CTRPv2 <-> PRISM matches by canonical BRD ID
ctrp_subset = dataset_frames.get("CTRPv2", pd.DataFrame())
prism_subset = dataset_frames.get("PRISM", pd.DataFrame())

brd_merged = ctrp_subset.merge(prism_subset, on="brd_canonical", suffixes=("_a", "_b"), how="inner")
brd_candidates_df = pd.DataFrame(
    {
        "dataset_a": brd_merged["dataset_a"],
        "identifier_a": brd_merged["identifier_a"],
        "compound_name_a": brd_merged["compound_name_a"],
        "dataset_b": brd_merged["dataset_b"],
        "identifier_b": brd_merged["identifier_b"],
        "compound_name_b": brd_merged["compound_name_b"],
        "compound_name_norm": brd_merged["compound_name_norm_a"],
        "brd_canonical": brd_merged["brd_canonical"],
        "match_method": "brd_exact",
        "target_a": brd_merged["target_a"],
        "target_b": brd_merged["target_b"],
        "moa_or_pathway_a": brd_merged["moa_or_pathway_a"],
        "moa_or_pathway_b": brd_merged["moa_or_pathway_b"],
    }
)

# Combine and de-duplicate candidate edges
candidates_df = pd.concat([name_candidates_df, brd_candidates_df], ignore_index=True)
candidates_df = candidates_df.drop_duplicates(
    subset=["dataset_a", "identifier_a", "dataset_b", "identifier_b", "match_method"]
)

print("\nCandidate overlap pairs by match method:")
print(candidates_df["match_method"].value_counts())

# Save for manual review/dedup curation
candidate_output = "../data/drug/drug_overlap_candidates.csv"
summary_output = "../data/drug/drug_overlap_summary.csv"
candidates_df.to_csv(candidate_output, index=False)
overlap_summary_df.to_csv(summary_output, index=False)

print(f"Saved candidate overlap pairs to: {candidate_output}")
print(f"Saved overlap summary to: {summary_output}")

display(candidates_df.head(20))


Pairwise overlap summary:


,dataset_a,dataset_b,name_overlap_count,brd_overlap_count
0,CTRPv2,GDSC,66,0
1,CTRPv2,PRISM,218,243
2,GDSC,PRISM,144,0



Candidate overlap pairs by match method:
match_method
name_exact    438
brd_exact     243
Name: count, dtype: int64
Saved candidate overlap pairs to: ../data/drug/drug_overlap_candidates.csv
Saved overlap summary to: ../data/drug/drug_overlap_summary.csv


,dataset_a,identifier_a,compound_name_a,dataset_b,identifier_b,compound_name_b,compound_name_norm,brd_canonical,match_method,target_a,target_b,moa_or_pathway_a,moa_or_pathway_b
0,CTRPv2,23151,tretinoin,GDSC,1009,Tretinoin,tretinoin,BRD-K64634304,name_exact,RARA;RARB;RARG,Retinoic acid,agonist of retinoid acid receptors,Other
1,CTRPv2,26956,paclitaxel,GDSC,1080,Paclitaxel,paclitaxel,BRD-A28746609,name_exact,NaN,Microtubule stabiliser,inhibitor of microtubule assembly,Mitosis
2,CTRPv2,26972,tamoxifen,GDSC,1199,Tamoxifen,tamoxifen,BRD-K93754473,name_exact,ESR1;ESR2,ESR1,modulator of estrogen receptors,Hormone-related
3,CTRPv2,27871,teniposide,GDSC,1809,Teniposide,teniposide,BRD-A35588707,name_exact,TOP2A;TOP2B,NaN,inhibitor of topoisomerase II,DNA replication
4,CTRPv2,30371,methotrexate,GDSC,1008,Methotrexate,methotrexate,BRD-K59456551,name_exact,DHFR,Antimetabolite,inhibitor of dihydrofolate reductase,DNA replication
5,CTRPv2,32620,cyclophosphamide,GDSC,1512,Cyclophosphamide,cyclophosphamide,BRD-A09722536,name_exact,NaN,Alkylating agent,DNA alkylator,DNA replication
6,CTRPv2,32622,dacarbazine,GDSC,1815,Dacarbazine,dacarbazine,BRD-K35520305,name_exact,NaN,CP11A,DNA alkylator,Other
7,CTRPv2,39782,piperlongumine,GDSC,1243,Piperlongumine,piperlongumine,BRD-K24132293,name_exact,NaN,Induces reactive oxygen species,natural product; modulator of ROS levels,Other
8,CTRPv2,44580,topotecan,GDSC,1808,Topotecan,topotecan,BRD-K55696337,name_exact,TOP1,TOP1,inhibitor of topoisomerase I,DNA replication
9,CTRPv2,50134,tanespimycin,GDSC,1026,Tanespimycin,tanespimycin,BRD-K81473043,name_exact,HSP90AA1,HSP90,inhibitor of HSP90,Protein stability and degradation


In [117]:
# Summary stats from overlap artifacts generated in the previous cell
from collections import Counter

# Reuse prepared objects from previous cell; rebuild only if needed
if "dataset_sets" not in globals():
    dataset_sets = {
        ds: set(sub["compound_name_norm"].dropna().unique())
        for ds, sub in drug_df.groupby("dataset")
    }

if "overlap_summary_df" not in globals():
    summary_rows = []
    datasets = sorted(dataset_sets.keys())
    for i, ds1 in enumerate(datasets):
        for ds2 in datasets[i + 1:]:
            set_a = dataset_sets[ds1]
            set_b = dataset_sets[ds2]
            summary_rows.append(
                {
                    "dataset_a": ds1,
                    "dataset_b": ds2,
                    "name_overlap_count": len(set_a & set_b),
                }
            )
    overlap_summary_df = pd.DataFrame(summary_rows)

# Unique counts
unique_per_dataset = (
    pd.Series({ds: len(compounds) for ds, compounds in dataset_sets.items()})
    .sort_values(ascending=False)
    .rename_axis("dataset")
    .reset_index(name="unique_compound_count")
)
all_compounds = set().union(*dataset_sets.values())
overall_unique_compounds = len(all_compounds)

print("Unique compounds per dataset:")
display(unique_per_dataset)
print(f"Overall unique compounds across all datasets (union): {overall_unique_compounds}")

# Add percentages to existing pairwise overlap counts
pairwise_overlap_df = overlap_summary_df.copy()
if "overlap_count" in pairwise_overlap_df.columns and "name_overlap_count" not in pairwise_overlap_df.columns:
    pairwise_overlap_df = pairwise_overlap_df.rename(columns={"overlap_count": "name_overlap_count"})

pairwise_overlap_df["set_size_a"] = pairwise_overlap_df["dataset_a"].map(lambda ds: len(dataset_sets[ds]))
pairwise_overlap_df["set_size_b"] = pairwise_overlap_df["dataset_b"].map(lambda ds: len(dataset_sets[ds]))
pairwise_overlap_df["union_size"] = pairwise_overlap_df.apply(
    lambda r: len(dataset_sets[r["dataset_a"]] | dataset_sets[r["dataset_b"]]), axis=1
)
pairwise_overlap_df["pct_of_a"] = 100 * pairwise_overlap_df["name_overlap_count"] / pairwise_overlap_df["set_size_a"]
pairwise_overlap_df["pct_of_b"] = 100 * pairwise_overlap_df["name_overlap_count"] / pairwise_overlap_df["set_size_b"]
pairwise_overlap_df["jaccard_pct"] = 100 * pairwise_overlap_df["name_overlap_count"] / pairwise_overlap_df["union_size"]

print("\nPairwise overlap percentages (name-based):")
display(pairwise_overlap_df[["dataset_a", "dataset_b", "name_overlap_count", "pct_of_a", "pct_of_b", "jaccard_pct"]])

# Membership distribution: in how many datasets each compound appears
membership_counts = Counter(
    sum(comp in compounds for compounds in dataset_sets.values())
    for comp in all_compounds
)
membership_summary_df = (
    pd.Series(membership_counts)
    .sort_index()
    .rename_axis("present_in_n_datasets")
    .reset_index(name="compound_count")
)
membership_summary_df["pct_of_overall_unique"] = (
    100 * membership_summary_df["compound_count"] / overall_unique_compounds
)

print("\nDistribution of compounds by number of datasets they appear in:")
display(membership_summary_df)

# Exact dataset-combination breakdown
combo_counts = Counter(
    " | ".join(sorted(ds for ds, compounds in dataset_sets.items() if comp in compounds))
    for comp in all_compounds
)
combo_breakdown_df = (
    pd.Series(combo_counts)
    .sort_values(ascending=False)
    .rename_axis("dataset_combination")
    .reset_index(name="compound_count")
)
combo_breakdown_df["pct_of_overall_unique"] = (
    100 * combo_breakdown_df["compound_count"] / overall_unique_compounds
)

print("\nExact dataset-combination breakdown:")
display(combo_breakdown_df)


Unique compounds per dataset:


,dataset,unique_compound_count
0,PRISM,6575
1,CTRPv2,545
2,GDSC,286


Overall unique compounds across all datasets (union): 7040

Pairwise overlap percentages (name-based):


,dataset_a,dataset_b,name_overlap_count,pct_of_a,pct_of_b,jaccard_pct
0,CTRPv2,GDSC,66,12.110092,23.076923,8.627451
1,CTRPv2,PRISM,218,40.000000,3.315589,3.158505
2,GDSC,PRISM,144,50.349650,2.190114,2.143814



Distribution of compounds by number of datasets they appear in:


,present_in_n_datasets,compound_count,pct_of_overall_unique
0,1,6736,95.681818
1,2,242,3.437500
2,3,62,0.880682



Exact dataset-combination breakdown:


,dataset_combination,compound_count,pct_of_overall_unique
0,PRISM,6275,89.133523
1,CTRPv2,323,4.588068
2,CTRPv2 | PRISM,156,2.215909
3,GDSC,138,1.960227
4,GDSC | PRISM,82,1.164773
5,CTRPv2 | GDSC | PRISM,62,0.880682
6,CTRPv2 | GDSC,4,0.056818


In [118]:
drugbank = pd.read_xml("/Users/selin/Desktop/OncoTox/full database.xml")
drugbank.head()

,type,created,updated,drugbank-id,name,description,cas-number,unii,state,groups,...,snp-adverse-drug-reactions,targets,enzymes,carriers,transporters,fda-label,msds,average-mass,monoisotopic-mass,calculated-properties
0,biotech,2005-06-13,2026-02-02,BIOD00024,Lepirudin,Lepirudin is a recombinant hirudin formed by 6...,138068-37-8,Y43GF64R34,solid,\n,...,NaN,\n,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,biotech,2005-06-13,2026-03-05,BIOD00071,Cetuximab,Cetuximab is a recombinant chimeric human/mous...,205923-56-4,PQX0D8J21J,liquid,\n,...,NaN,\n,NaN,NaN,NaN,//s3-us-west-2.amazonaws.com/drugbank/fda_labe...,//s3-us-west-2.amazonaws.com/drugbank/msds/DB0...,NaN,NaN,NaN
2,biotech,2005-06-13,2026-03-05,BIOD00001,Dornase alfa,Dornase alfa is a biosynthetic form of human d...,143831-71-4,953A26OA1Y,liquid,\n,...,NaN,\n,NaN,NaN,NaN,//s3-us-west-2.amazonaws.com/drugbank/fda_labe...,//s3-us-west-2.amazonaws.com/drugbank/msds/DB0...,NaN,NaN,NaN
3,biotech,2005-06-13,2026-03-05,BIOD00084,Denileukin diftitox,Denileukin diftitox is an IL2-receptor-directe...,173146-27-5,25E79B5CTM,liquid,\n,...,NaN,\n,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,biotech,2005-06-13,2026-03-05,BIOD00052,Etanercept,Dimeric fusion protein consisting of the extra...,185243-69-0,OP401G7OJC,liquid,\n,...,NaN,\n,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [119]:
# Overlap between each dataset and DrugBank
import re


def normalize_drug_text(value):
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    # Remove punctuation/spacing differences (e.g., "5-fluorouracil" vs "5 fluorouracil")
    text = re.sub(r"[^a-z0-9]+", "", text)
    return text


# Ensure drug catalog and DrugBank are available
if "all_drugs_df" not in globals():
    all_drugs_df = pd.read_csv("../data/all_sources_drug_catalog.csv")
if "drugbank" not in globals():
    drugbank = pd.read_xml("/Users/selin/Desktop/OncoTox/full database.xml")

# Prepare DrugBank keys
drugbank_ref = drugbank[["drugbank-id", "name", "groups", "cas-number", "unii"]].copy()
drugbank_ref["drugbank_name_norm"] = drugbank_ref["name"].map(normalize_drug_text)
drugbank_ref = drugbank_ref[drugbank_ref["drugbank_name_norm"] != ""].drop_duplicates(
    subset=["drugbank_name_norm"]
)
drugbank_name_set = set(drugbank_ref["drugbank_name_norm"])

# Prepare source compound keys (name + optional synonym expansion)
catalog = all_drugs_df.copy()
catalog["compound_name_norm"] = catalog["compound_name"].map(normalize_drug_text)
catalog = catalog[catalog["compound_name_norm"] != ""]

# Optional synonym tokens from PRISM/other sources if present
synonym_rows = []
if "synonyms" in catalog.columns:
    syn_source = catalog.dropna(subset=["synonyms"])[["identifier", "dataset", "compound_name", "synonyms"]].copy()
    for _, row in syn_source.iterrows():
        parts = re.split(r"[|;,]", str(row["synonyms"]))
        for syn in parts:
            syn_norm = normalize_drug_text(syn)
            if syn_norm:
                synonym_rows.append(
                    {
                        "identifier": row["identifier"],
                        "dataset": row["dataset"],
                        "compound_name": row["compound_name"],
                        "compound_name_norm": syn_norm,
                        "match_basis": "synonym",
                    }
                )

name_rows = catalog[["identifier", "dataset", "compound_name", "compound_name_norm"]].copy()
name_rows["match_basis"] = "name"

expanded_catalog = pd.concat(
    [name_rows, pd.DataFrame(synonym_rows)], ignore_index=True, sort=False
).drop_duplicates(subset=["dataset", "identifier", "compound_name_norm", "match_basis"])

# Match to DrugBank by normalized name key
matched = expanded_catalog[expanded_catalog["compound_name_norm"].isin(drugbank_name_set)].copy()
unmatched = expanded_catalog[~expanded_catalog["compound_name_norm"].isin(drugbank_name_set)].copy()

# Keep one representative match per dataset+identifier
matched_unique = matched.sort_values(by=["dataset", "identifier", "match_basis"]).drop_duplicates(
    subset=["dataset", "identifier"], keep="first"
)

# Attach DrugBank metadata to matched entries
matched_with_db = matched_unique.merge(
    drugbank_ref,
    left_on="compound_name_norm",
    right_on="drugbank_name_norm",
    how="left",
)

# Summary per dataset
total_unique_per_dataset = (
    catalog.groupby("dataset")["identifier"].nunique().rename("dataset_unique_compounds")
)
matched_unique_per_dataset = (
    matched_unique.groupby("dataset")["identifier"].nunique().rename("matched_in_drugbank")
)

overlap_vs_drugbank_df = (
    pd.concat([total_unique_per_dataset, matched_unique_per_dataset], axis=1)
    .fillna(0)
    .reset_index()
)
overlap_vs_drugbank_df["matched_in_drugbank"] = overlap_vs_drugbank_df[
    "matched_in_drugbank"
].astype(int)
overlap_vs_drugbank_df["pct_dataset_in_drugbank"] = (
    100
    * overlap_vs_drugbank_df["matched_in_drugbank"]
    / overlap_vs_drugbank_df["dataset_unique_compounds"]
)

# Overall stats
overall_unique = int(catalog["identifier"].nunique())
overall_matched = int(matched_unique["identifier"].nunique())
overall_pct = 100 * overall_matched / overall_unique if overall_unique else 0

print("Overlap with DrugBank by dataset:")
display(overlap_vs_drugbank_df.sort_values("dataset"))
print(
    f"Overall matched unique compounds: {overall_matched}/{overall_unique} ({overall_pct:.2f}%)"
)

# Save review tables
matched_out = "../data/drug/drugbank_overlap_matches.csv"
unmatched_out = "../data/drug/drugbank_overlap_unmatched.csv"
summary_out = "../data/drug/drugbank_overlap_summary.csv"

matched_with_db.to_csv(matched_out, index=False)
unmatched.drop_duplicates(subset=["dataset", "identifier"]).to_csv(unmatched_out, index=False)
overlap_vs_drugbank_df.to_csv(summary_out, index=False)

print(f"Saved matched table: {matched_out}")
print(f"Saved unmatched table: {unmatched_out}")
print(f"Saved summary table: {summary_out}")

display(matched_with_db[["dataset", "identifier", "compound_name", "match_basis", "drugbank-id", "name", "groups"]].head(20))


Overlap with DrugBank by dataset:


,dataset,dataset_unique_compounds,matched_in_drugbank,pct_dataset_in_drugbank
0,CTRPv2,545,173,31.743119
1,GDSC,295,118,40.000000
2,PRISM,6575,3483,52.973384


Overall matched unique compounds: 3774/7415 (50.90%)
Saved matched table: ../data/drug/drugbank_overlap_matches.csv
Saved unmatched table: ../data/drug/drugbank_overlap_unmatched.csv
Saved summary table: ../data/drug/drugbank_overlap_summary.csv


,dataset,identifier,compound_name,match_basis,drugbank-id,name,groups
0,CTRPv2,23151,tretinoin,name,APRD00362,Tretinoin,\n
1,CTRPv2,23256,betulinic acid,name,DB05910,Betulinic Acid,\n
2,CTRPv2,24173,phloretin,name,DB07810,Phloretin,\n
3,CTRPv2,24197,thalidomide,name,APRD01251,Thalidomide,\n
4,CTRPv2,25036,gossypol,name,DB13044,Gossypol,\n
5,CTRPv2,25334,chlorambucil,name,APRD00115,Chlorambucil,\n
6,CTRPv2,25344,fluorouracil,name,EXPT03204,Fluorouracil,\n
7,CTRPv2,25393,isoliquiritigenin,name,EXPT01707,Isoliquiritigenin,\n
8,CTRPv2,26870,cimetidine,name,APRD00568,Cimetidine,\n
9,CTRPv2,26874,azacitidine,name,APRD00809,Azacitidine,\n


In [121]:
# Target-value coverage for overlapping SCP542 cell lines vs each dataset's own cell lines
# Assumptions:
# - GDSC target metric: LN_IC50
# - CTRPv2 target metric: cpd_avg_pv (viability proxy)
# - PRISM target metric: values in Extended Primary Data Matrix (per compound x depmap cell line)

# SCP542 normalized cell-line names used for overlap checks
scp542_overlap_set = set(
    scp542_scrna["Cell_line"].dropna().astype(str).str.strip().str.lower().unique()
)

# ---------------------
# GDSC: row-wise LN_IC50
# ---------------------
gdsc_eval = gdsc_bulk.copy()
gdsc_eval["cell_line_norm"] = (
    gdsc_eval["CELL_LINE_NAME"].astype(str).str.strip().str.lower()
)

gdsc_non_null_own = int(gdsc_eval["LN_IC50"].notna().sum())
gdsc_overlap_mask = gdsc_eval["cell_line_norm"].isin(scp542_overlap_set)
gdsc_total_overlap_rows = int(gdsc_overlap_mask.sum())
gdsc_non_null_overlap = int(
    gdsc_eval.loc[gdsc_overlap_mask, "LN_IC50"].notna().sum()
)

# -----------------------------------------
# CTRPv2: map experiment_id -> ccl_name, use cpd_avg_pv
# -----------------------------------------
ctrp_values = pd.read_csv(
    "/Users/selin/Desktop/OncoTox/metadata/CTRPv2.0_2015_ctd2_ExpandedDataset/v20.data.per_cpd_post_qc.txt",
    sep="\t",
    usecols=["experiment_id", "cpd_avg_pv"],
)
ctrp_exp_meta = pd.read_csv(
    "/Users/selin/Desktop/OncoTox/metadata/CTRPv2.0_2015_ctd2_ExpandedDataset/v20.meta.per_experiment.txt",
    sep="\t",
    usecols=["experiment_id", "master_ccl_id"],
)
ctrp_cell_meta = pd.read_csv(
    "/Users/selin/Desktop/OncoTox/metadata/CTRPv2.0_2015_ctd2_ExpandedDataset/v20.meta.per_cell_line.txt",
    sep="\t",
    usecols=["master_ccl_id", "ccl_name"],
)

ctrp_eval = (
    ctrp_values.merge(ctrp_exp_meta, on="experiment_id", how="left")
    .merge(ctrp_cell_meta, on="master_ccl_id", how="left")
)
ctrp_eval["cell_line_norm"] = (
    ctrp_eval["ccl_name"].astype(str).str.strip().str.lower()
)

ctrp_non_null_own = int(ctrp_eval["cpd_avg_pv"].notna().sum())
ctrp_overlap_mask = ctrp_eval["cell_line_norm"].isin(scp542_overlap_set)
ctrp_total_overlap_rows = int(ctrp_overlap_mask.sum())
ctrp_non_null_overlap = int(
    ctrp_eval.loc[ctrp_overlap_mask, "cpd_avg_pv"].notna().sum()
)

# ---------------------------------------------------------
# PRISM: matrix values across depmap columns (non-null cells)
# ---------------------------------------------------------
prism_matrix = pd.read_csv(
    "/Users/selin/Desktop/OncoTox/metadata/PRISM_REPURPOSED/Repurposing_Public_24Q2_Extended_Primary_Data_Matrix.csv"
)
prism_cl_meta = pd.read_csv(
    "/Users/selin/Desktop/OncoTox/metadata/PRISM_REPURPOSED/Repurposing_Public_24Q2_Cell_Line_Meta_Data.csv"
)

# Map SCP542 names -> DepMap IDs through PRISM cell-line metadata
prism_cl_meta["base_ccle_name"] = (
    prism_cl_meta["ccle_name"].astype(str).str.split("_").str[0].str.strip().str.lower()
)
overlap_depmap_ids = set(
    prism_cl_meta.loc[
        prism_cl_meta["base_ccle_name"].isin(scp542_overlap_set), "depmap_id"
    ]
    .dropna()
    .astype(str)
)

value_cols = [c for c in prism_matrix.columns if c != "Unnamed: 0"]
overlap_value_cols = [c for c in value_cols if c in overlap_depmap_ids]

prism_non_null_own = int(prism_matrix[value_cols].notna().sum().sum())
prism_total_overlap_rows = int(prism_matrix[overlap_value_cols].size)
prism_non_null_overlap = int(prism_matrix[overlap_value_cols].notna().sum().sum())

# ---------------------
# Summary table
# ---------------------
target_coverage_df = pd.DataFrame(
    [
        {
            "dataset": "GDSC",
            "target_metric": "LN_IC50",
            "non_null_rows_own_cell_lines": gdsc_non_null_own,
            "rows_in_overlap_subset": gdsc_total_overlap_rows,
            "non_null_rows_overlap_scp542": gdsc_non_null_overlap,
        },
        {
            "dataset": "CTRPv2",
            "target_metric": "cpd_avg_pv (viability)",
            "non_null_rows_own_cell_lines": ctrp_non_null_own,
            "rows_in_overlap_subset": ctrp_total_overlap_rows,
            "non_null_rows_overlap_scp542": ctrp_non_null_overlap,
        },
        {
            "dataset": "PRISM",
            "target_metric": "extended primary matrix value (viability proxy)",
            "non_null_rows_own_cell_lines": prism_non_null_own,
            "rows_in_overlap_subset": prism_total_overlap_rows,
            "non_null_rows_overlap_scp542": prism_non_null_overlap,
        },
    ]
)
target_coverage_df["pct_overlap_vs_own"] = (
    100
    * target_coverage_df["non_null_rows_overlap_scp542"]
    / target_coverage_df["non_null_rows_own_cell_lines"]
)

# This is the "against SCP overlap subset" percentage: non-null within overlap-only rows
# (i.e., completeness of response values among overlap rows)
target_coverage_df["pct_non_null_within_overlap_subset"] = (
    100
    * target_coverage_df["non_null_rows_overlap_scp542"]
    / target_coverage_df["rows_in_overlap_subset"]
)

display(target_coverage_df)


,dataset,target_metric,non_null_rows_own_cell_lines,rows_in_overlap_subset,non_null_rows_overlap_scp542,pct_overlap_vs_own,pct_non_null_within_overlap_subset
0,GDSC,LN_IC50,242036,34738,34738,14.352410,100.000000
1,CTRPv2,cpd_avg_pv (viability),7227951,1521028,1521028,21.043696,100.000000
2,PRISM,extended primary matrix value (viability proxy),4213048,1235780,1210432,28.730553,97.948826
